# 03 — Nano 9B v2 one-GPU Text2SQL LoRA workshop

This is the default practical path: 2,048 direct-SQL training rows, 2,048-token packing, rank 32, GBS 32, and at most 32 steps on one GPU. It targets a warm-cache end-to-end workshop run within about an hour, but that target remains unverified until the exact Brev SKU is rehearsed. The pinned recipe supports one H100; the repository conservatively recommends an 80 GB A100/H100 and treats 48 GB as experimental.

The proof boundary is unchanged: compare the merged adapter with the **matching**
local BF16 base on the same frozen 100 BIRD Mini-Dev IDs. Training loss is a
diagnostic; executable held-out SQL is the success metric.

**Required runtime:** use the NeMo container Jupyter opened by
`launchable/setup.sh` through the Secure Link on host port **8889**.


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

from nemotron_ft_lab.model_profiles import get_model_profile

PROFILE = get_model_profile('nano9b_workshop')
ARTIFACTS_DIR = Path(os.environ.get('NEMOTRON_ARTIFACTS_DIR', ROOT / 'artifacts')).expanduser().resolve()
EVAL_DATA_DIR = ARTIFACTS_DIR / 'data/bird-text2sql'
TRAIN_DATA_DIR = EVAL_DATA_DIR / 'profiles' / PROFILE.name
EVALUATION_DIR = ARTIFACTS_DIR / 'evaluation' / PROFILE.artifact_slug
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_PATH = EVALUATION_DIR / 'baseline_local_bf16_text2sql.json'
PEFT_REPORT_PATH = EVALUATION_DIR / 'peft_text2sql.json'
CHECKPOINT_ROOT = Path('/workspace/storage/checkpoints')
MEGATRON_BASE = CHECKPOINT_ROOT / PROFILE.megatron_checkpoint_name
LORA_ROOT = CHECKPOINT_ROOT / PROFILE.lora_checkpoint_name
MERGED_MODEL = CHECKPOINT_ROOT / PROFILE.merged_checkpoint_name

RUN_TRAINING = True
RESUME_IF_AVAILABLE = True
RUN_MERGE = True
RUN_EVALUATION = True
print(json.dumps(PROFILE.as_dict(), indent=2))
print('Python:', sys.executable)
print('Artifacts:', ARTIFACTS_DIR)


## 1. Verify hardware, matching baseline, and profile-specific training data

The training manifest binds the model revision, native system prompt, reasoning
mode, row count, and data hash. The driver rejects a mismatched manifest.


In [ ]:
subprocess.run([
    sys.executable, 'scripts/preflight.py', '--profile', 'peft',
    '--model-profile', PROFILE.name,
], check=True)
if not BASELINE_PATH.exists():
    raise RuntimeError(
        f'Run Notebook 02 with MODEL_PROFILE_NAME={PROFILE.name!r} first: {BASELINE_PATH}'
    )
subprocess.run([
    sys.executable, 'scripts/prepare_text2sql.py',
    '--model-profile', PROFILE.name,
    '--output-dir', str(TRAIN_DATA_DIR), '--training-only',
    '--max-train-samples', str(PROFILE.default_train_samples),
    '--max-sequence-length', '2048',
], check=True)

import torch

baseline = json.loads(BASELINE_PATH.read_text())
if baseline.get('model_profile') != PROFILE.name:
    raise RuntimeError('Baseline model profile does not match this LoRA run.')
train_manifest = json.loads((TRAIN_DATA_DIR / 'training_manifest.json').read_text())
print('Frozen baseline execution accuracy:', baseline['execution_accuracy'])
print('Training rows:', train_manifest['training_examples'])
print('Training mix:', train_manifest['source_distribution'])
print('Reasoning traces enabled:', train_manifest['include_reasoning'])
print('Train/eval policy:', train_manifest['split_policy'])


## 2. Download and convert the pinned BF16 checkpoint once

Both the Hugging Face snapshot and converted Megatron checkpoint are reused.
A cold workshop should prefetch these in setup rather than spending participant
time on network and conversion work.


In [ ]:
from huggingface_hub import snapshot_download

stage_times = {}
started = time.perf_counter()
PINNED_HF_MODEL = Path(snapshot_download(
    repo_id=PROFILE.model_id, revision=PROFILE.revision,
))
convert_cmd = [
    sys.executable, 'scripts/convert_checkpoint.py',
    '--model-profile', PROFILE.name,
    '--hf-model', PROFILE.model_id, '--revision', PROFILE.revision,
    '--output', str(MEGATRON_BASE),
]
if RUN_TRAINING:
    subprocess.run(convert_cmd, check=True)
else:
    print('Would run:', ' '.join(convert_cmd))
stage_times['snapshot_and_conversion_minutes'] = (time.perf_counter() - started) / 60
print(json.dumps(stage_times, indent=2))


## 3. Train with the pinned model-family recipe

Nano uses Megatron-Bridge's one-GPU H100 BF16 recipe and saves every eight
steps. Lightning uses its official model-specific LoRA targets and expert
parallelism. `--resume` is added only when a compatible checkpoint and run
contract already exist; changing the experiment requires a new output path.


In [ ]:
VISIBLE_GPUS = torch.cuda.device_count()
N_GPUS = int(os.environ.get('NEMOTRON_PEFT_NUM_GPUS', '1'))
if N_GPUS not in PROFILE.peft_world_sizes:
    raise RuntimeError(f'{PROFILE.name} supports PEFT world sizes {PROFILE.peft_world_sizes}, not {N_GPUS}.')
if N_GPUS > VISIBLE_GPUS:
    raise RuntimeError(f'Requested {N_GPUS} GPU(s), but only {VISIBLE_GPUS} are visible.')
train_cmd = [
    'torchrun', f'--nproc-per-node={N_GPUS}', 'scripts/train_peft.py',
    '--model-profile', PROFILE.name,
    '--hf-model', PROFILE.model_id, '--revision', PROFILE.revision,
    '--megatron-checkpoint', str(MEGATRON_BASE),
    '--data-dir', str(TRAIN_DATA_DIR), '--output-dir', str(LORA_ROOT),
    '--sequence-length', '2048',
    '--global-batch-size', str(PROFILE.default_global_batch_size),
    '--max-steps', str(PROFILE.default_max_steps),
    '--learning-rate', '1e-4', '--lora-rank', '32',
]
marker = LORA_ROOT / 'latest_checkpointed_iteration.txt'
if RESUME_IF_AVAILABLE and marker.exists():
    train_cmd.append('--resume')
print(f'Using {N_GPUS}/{VISIBLE_GPUS} GPU(s).')
print('Launch:', ' '.join(train_cmd))
if RUN_TRAINING:
    started = time.perf_counter()
    subprocess.run(train_cmd, check=True)
    stage_times['lora_training_minutes'] = (time.perf_counter() - started) / 60


In [ ]:
marker = LORA_ROOT / 'latest_checkpointed_iteration.txt'
if RUN_TRAINING:
    if not marker.exists():
        raise RuntimeError(f'Missing adapter marker: {marker}')
    latest_step = int(marker.read_text().strip())
    adapter_checkpoint = LORA_ROOT / f'iter_{latest_step:07d}'
    if not adapter_checkpoint.is_dir():
        raise RuntimeError(f'Missing adapter checkpoint: {adapter_checkpoint}')
    print('Saved adapter:', adapter_checkpoint)
else:
    adapter_checkpoint = Path('/path/to/adapter')


## 4. Merge the adapter to a standard Hugging Face checkpoint


In [ ]:
BRIDGE_DIR = Path(os.environ.get('MEGATRON_BRIDGE_DIR', '/workspace/storage/Megatron-Bridge'))
merge_cmd = [
    'torchrun', '--nproc-per-node=1', str(BRIDGE_DIR / 'examples/peft/merge_lora.py'),
    '--lora-checkpoint', str(adapter_checkpoint),
    '--hf-model-path', str(PINNED_HF_MODEL),
    '--output', str(MERGED_MODEL), '--cpu',
]
print('Merge:', ' '.join(merge_cmd))
if RUN_MERGE:
    started = time.perf_counter()
    subprocess.run(merge_cmd, check=True)
    if not (MERGED_MODEL / 'config.json').exists():
        raise RuntimeError('Merge completed without config.json.')
    stage_times['cpu_merge_minutes'] = (time.perf_counter() - started) / 60


## 5. Evaluate the unchanged 100-row holdout with vLLM


In [ ]:
INFERENCE_GPUS = int(os.environ.get('NEMOTRON_INFERENCE_GPUS', '1'))
eval_cmd = [
    sys.executable, 'scripts/evaluate_vllm.py',
    '--model-profile', PROFILE.name,
    '--model', str(MERGED_MODEL), '--revision', '',
    '--data-dir', str(EVAL_DATA_DIR), '--output', str(PEFT_REPORT_PATH),
    '--run-type', f'lora-peft-text2sql-{PROFILE.name}',
    '--tensor-parallel-size', str(INFERENCE_GPUS),
]
if RUN_EVALUATION:
    started = time.perf_counter()
    subprocess.run(eval_cmd, check=True)
    stage_times['merged_evaluation_minutes'] = (time.perf_counter() - started) / 60
tuned = json.loads(PEFT_REPORT_PATH.read_text())
print('Measured stages:', json.dumps(stage_times, indent=2))


In [ ]:
from nemotron_ft_lab.evaluation import paired_execution_comparison

comparison = paired_execution_comparison(baseline, tuned)
comparison.update({
    'model_profile': PROFILE.name,
    'baseline_execution_accuracy': baseline['execution_accuracy'],
    'peft_execution_accuracy': tuned['execution_accuracy'],
    'baseline_sql_valid_rate': baseline['sql_valid_rate'],
    'peft_sql_valid_rate': tuned['sql_valid_rate'],
})
print(json.dumps(comparison, indent=2))
if comparison['absolute_execution_accuracy_gain'] <= 0:
    print('No held-out execution gain was demonstrated. Do not claim success from loss alone.')

baseline_by_id = {row['example_id']: row for row in baseline['rows']}
tuned_by_id = {row['example_id']: row for row in tuned['rows']}
changed = [
    item for item in baseline_by_id
    if baseline_by_id[item]['execution_correct'] != tuned_by_id[item]['execution_correct']
]
for item in changed[:8]:
    before, after = baseline_by_id[item], tuned_by_id[item]
    print()
    print('Q:', after['question'])
    print('Gold:', after['expected_sql'])
    print('Base:', before['generated'], 'correct=', before['execution_correct'])
    print('LoRA:', after['generated'], 'correct=', after['execution_correct'])


## 6. Optional context: compare with hosted targets on shared IDs


In [ ]:
cloud_results = {}
for path in sorted((ARTIFACTS_DIR / 'evaluation').glob('baseline_api_*_text2sql_*.json')):
    cloud = json.loads(path.read_text())
    ids = [row['example_id'] for row in cloud['rows']]
    if not set(ids).issubset(tuned_by_id):
        continue
    tuned_shared = {'rows': [tuned_by_id[item] for item in ids]}
    result = paired_execution_comparison(cloud, tuned_shared)
    result.update({
        'cloud_model': cloud['model'],
        'cloud_execution_accuracy': cloud['execution_accuracy'],
        'tuned_execution_accuracy_on_shared_ids': (
            sum(row['execution_correct'] for row in tuned_shared['rows']) / len(ids)
        ),
        'interpretation': f'tuned {PROFILE.name} minus hosted target on shared IDs',
    })
    cloud_results[path.name] = result
    print(path.name, json.dumps(result, indent=2))
target_path = EVALUATION_DIR / 'peft_vs_cloud_text2sql.json'
target_path.write_text(json.dumps(cloud_results, indent=2) + '\n')
print('Saved:', target_path)


## What counts as success

A positive base-to-LoRA execution delta demonstrates task adaptation for this
run. A paired 95% confidence interval above zero is stronger evidence. Matching
a hosted larger model on only 25 shared rows is useful context, not a general
model ranking. Always report hardware, cold/warm cache state, stage times, and
the exact profile beside the accuracy result.
